In [ ]:
!pip install -q gradio transformers torch keybert sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.9 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import re
import torch
from transformers import pipeline
from keybert import KeyBERT


# ============================================
# LOAD MODELS
# ============================================

print("Loading Speech Recognition model...")

speech_model = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny"
)

print("Loading Sentiment Analysis model...")

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Loading Keyword Extraction model...")

keyword_model = KeyBERT("all-MiniLM-L6-v2")

print("All models loaded successfully!")


# ============================================
# TEXT ANALYSIS
# ============================================

def analyze_text(text):

    if not text or not text.strip():
        return (
            "No text detected.",
            "No sentiment.",
            "No keywords.",
            0,
            0
        )

    text = text.strip()

    # Word count
    words = text.split()
    word_count = len(words)

    # Sentence count
    sentences = re.split(r'[.!?]+', text)
    sentences = [
        s.strip()
        for s in sentences
        if s.strip()
    ]

    sentence_count = len(sentences)

    # Sentiment
    sentiment_result = sentiment_model(text[:512])[0]

    label = sentiment_result["label"]
    score = sentiment_result["score"]

    sentiment = f"{label} ({score * 100:.2f}% confidence)"

    # Keywords
    keywords = keyword_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=8
    )

    keyword_list = ", ".join(
        [keyword for keyword, score in keywords]
    )

    if not keyword_list:
        keyword_list = "No keywords found."

    return (
        text,
        sentiment,
        keyword_list,
        word_count,
        sentence_count
    )


# ============================================
# SPEECH PROCESSING
# ============================================

def process_speech(audio):

    if audio is None:
        return (
            "Please record your voice.",
            "No sentiment.",
            "No keywords.",
            0,
            0
        )

    # Convert speech to text
    result = speech_model(audio)

    text = result["text"]

    # Analyze converted text
    return analyze_text(text)


# ============================================
# GRADIO INTERFACE
# ============================================

with gr.Blocks(
    title="Speech-to-Text & Text Analyzer"
) as app:

    gr.Markdown(
        """
        # 🎙️ Speech-to-Text + Text Analysis

        Speak using your microphone and the application will:

        🎤 Convert speech into text
        😊 Analyze sentiment
        🔑 Extract keywords
        📝 Count words
        📌 Count sentences
        """
    )

    # Microphone
    audio_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎤 Record Your Voice"
    )

    analyze_button = gr.Button(
        "🔍 Analyze Speech",
        variant="primary"
    )

    gr.Markdown("## 📊 Analysis Results")

    text_output = gr.Textbox(
        label="📝 Converted Speech",
        lines=5
    )

    sentiment_output = gr.Textbox(
        label="😊 Sentiment"
    )

    keywords_output = gr.Textbox(
        label="🔑 Keywords"
    )

    with gr.Row():

        word_output = gr.Number(
            label="📝 Word Count"
        )

        sentence_output = gr.Number(
            label="📌 Sentence Count"
        )

    analyze_button.click(
        fn=process_speech,
        inputs=audio_input,
        outputs=[
            text_output,
            sentiment_output,
            keywords_output,
            word_output,
            sentence_output
        ]
    )


# ============================================
# LAUNCH
# ============================================

app.launch(share=True)

Loading Speech Recognition model...


config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

All models loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e8db14598697c83516.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
